# Decision-making under ambiguity and poverty (DMAP)

A notebook that gives an overview and does a test run of the agent-based model for decision-making under ambiguity and poverty (DMAP).

`gamma` = utility parameter

`beta` = optimism/pessimism

`alpha` = likelihood sensitvity

`p` = probability of being caught

# agents.py

Constructing individual decision-maker and the rules they follow

In [3]:
import mesa
import numpy as np
from scipy.stats import bernoulli
import matplotlib.pyplot as plt
import seaborn as sns
import cognitive_functions as cf


In [62]:
class decision_maker(mesa.Agent):
    """An agent that chooses between two options based on expected utility."""
    def __init__(self, unique_id, model, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha):
        """ Create a new decision maker agent.
        Args:
            unique_id: Unique identifier for the agent.
            model: Reference to the model this agent belongs to.
            gamma: Utility parameter.
            reward_rb: Reward for rule breaking.
            reward_rf: Reward for following rules.
            cost_rb: Cost of rule breaking.
            wealth: Wealth of the agent.
            p: Probability of being caught for rule-breaking.
            beta: Optimism/pessimism.
            alpha: Likelihood sensitivity.
        """
        super().__init__(model)
        self.gamma = gamma
        self.reward_rb = reward_rb
        self.reward_rf = reward_rf
        self.cost_rb = cost_rb
        self.p = p
        self.beta = beta
        self.alpha = alpha
        self.wealth = wealth

    def compute_expected_utilities(self):
        """Choose an option based on expected utility."""
        self.EU_rule_break = cf.EU_rule_break(self.gamma, self.reward_rb, self.cost_rb, self.p, self.beta, self.alpha, self.wealth)  # EU of rule breaking
        self.EU_follow_rules = cf.EU_follow_rules(self.gamma, self.reward_rf, self.wealth) # EU of following rules
        probs = cf.softmax(self.EU_rule_break, self.EU_follow_rules)
        
        rb_choice = bernoulli.rvs(probs[0])  # Bernoulli trial for rule breaking choice
        if rb_choice==1:
            print(f"Agent {self.unique_id} chose to break the rules with probability {probs[0]}")
        else:
            print(f"Agent {self.unique_id} chose to follow the rules with probability {probs[1]}")
        return rb_choice
    
    def step(self):
        return self.compute_expected_utilities()  # Call the utility computation method

# model.py

Creating model environment to run the ABM.

In [65]:
from mesa import Model
from mesa.datacollection import DataCollector
import numpy as np

# from .agents import decision_maker

class DMAP_model(Model):
    """A model with a number of decision makers."""
    
    def __init__(self, N, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha):
        super().__init__()
        self.num_agents = N
        self.gamma = gamma
        self.reward_rb = reward_rb
        self.reward_rf = reward_rf
        self.cost_rb = cost_rb
        self.wealth = wealth
        self.p = p
        self.beta = beta
        self.alpha = alpha

        self.schedule = self.agents.shuffle_do(self)
        self.datacollector = mesa.DataCollector(
            agent_reporters={"Choice": "step"}
        )

        # Create agents
        for i in range(self.num_agents):
            a = decision_maker(i, self, self.gamma, self.reward_rb, self.reward_rf,
                              self.cost_rb, self.wealth, self.p, self.beta, self.alpha)
            self.schedule.add(a)
    
    def step(self):
        """Advance the model by one step."""
        self.datacollector.collect(self)
        # Activate all agents in random order
        self.agents.do("step")
        


## Run the model

In [66]:
test_model = DMAP_model(10, gamma=0.5, reward_rb=1000, reward_rf=50, cost_rb=25, wealth=50, p=0.05, beta=1.9, alpha=0.5)
test_model.step()

Agent 1 chose to break the rules with probability 0.5005103119093428
Agent 2 chose to break the rules with probability 0.5005103119093428
Agent 3 chose to break the rules with probability 0.5005103119093428
Agent 4 chose to break the rules with probability 0.5005103119093428
Agent 5 chose to follow the rules with probability 0.4994896880906573
Agent 6 chose to break the rules with probability 0.5005103119093428
Agent 7 chose to follow the rules with probability 0.4994896880906573
Agent 8 chose to break the rules with probability 0.5005103119093428
Agent 9 chose to follow the rules with probability 0.4994896880906573
Agent 10 chose to break the rules with probability 0.5005103119093428


## Visualization

In [ ]:
agent_wealth = [a.wealth for a in model.agents]
# Create a histogram with seaborn
g = sns.histplot(agent_wealth, discrete=True)
g.set(
    title="Wealth distribution", xlabel="Wealth", ylabel="number of agents"
);  # The semicolon is just to avoid printing the object representation